# 11 — KalmanNet Adaptive Filtering Training (Phase 4 — Fixed NIO Inputs)

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Section 19:** Use KalmanNet-style learning for adaptive filtering.
> Adapt covariance, measurement confidence, and dynamic filtering gain.
> Strict prohibition of arbitrary fixed gains ($K = 0.80$).

## Phase 4 Change
KalmanNet uses NIO velocity outputs as its measurement signal.
In v1, the NIO uncertainty was overflowed (σ=8,508,032), corrupting KalmanNet's measurement confidence.
In this v2 training run, KalmanNet is trained on the **fixed NIO** checkpoint (BoundedLogVarHead),
giving it physical velocity measurements with realistic uncertainties.

| Aspect | v1 | v2 (this notebook) |
|--------|----|-----------------|
| NIO source | `checkpoints/inertial_odometry/` (σ=8.5M) | `checkpoints/nio_fixed/` (σ∈[0.08,91m]) |
| KalmanNet checkpoint dir | `checkpoints/kalmannet/` | `checkpoints/kalmannet_fixed_input/` |
| Training provenance | Not recorded | Full metadata in results JSON |

In [ ]:
import os, sys, time, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.kalmannet import KalmanNetNN
from src.datasets.kalmannet_dataset import KalmanNetDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Hardware: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

plots_dir = PROJECT_ROOT / 'plots' / 'kalmannet'
plots_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── NIO Checkpoint Selection (Phase 4) ───────────────────────────────────────
# Priority: nio_fixed (v2 — BoundedLogVarHead) > baseline (v1 fallback)
FIXED_NIO_CKPT    = PROJECT_ROOT / 'checkpoints' / 'nio_fixed'         / 'nio_fixed_best.pt'
BASELINE_NIO_CKPT = PROJECT_ROOT / 'checkpoints' / 'inertial_odometry' / 'inertial_odometry_best.pt'

if FIXED_NIO_CKPT.exists():
    io_ckpt_str = str(FIXED_NIO_CKPT)
    nio_source  = 'nio_fixed_v2 (BoundedLogVarHead — sigma FIXED)'
    ckpt_dir    = PROJECT_ROOT / 'checkpoints' / 'kalmannet_fixed_input'
    results_tag = 'kalmannet_fixed_input'
elif BASELINE_NIO_CKPT.exists():
    io_ckpt_str = str(BASELINE_NIO_CKPT)
    nio_source  = 'nio_baseline_v1 (FALLBACK — fixed NIO not yet available)'
    ckpt_dir    = PROJECT_ROOT / 'checkpoints' / 'kalmannet'
    results_tag = 'kalmannet_baseline_input'
    print('[WARN] Fixed NIO not found. Using baseline. Run notebook 09 on GPU for best results.')
else:
    io_ckpt_str = None
    nio_source  = 'NO_NIO_CHECKPOINT'
    ckpt_dir    = PROJECT_ROOT / 'checkpoints' / 'kalmannet'
    results_tag = 'kalmannet_no_nio'
    print('[WARN] No NIO checkpoint found. KalmanNet will use raw IMU integration only.')

ckpt_dir.mkdir(parents=True, exist_ok=True)
print(f'NIO source   : {nio_source}')
print(f'NIO ckpt     : {io_ckpt_str}')
print(f'KalmanNet dir: {ckpt_dir}')
print(f'Results tag  : {results_tag}')

In [ ]:
print('Initializing Train & Validation Datasets for KalmanNet...')
train_ds = KalmanNetDataset(
    split='train',
    seq_len=50,
    stride=20,
    io_checkpoint_path=io_ckpt_str
)

val_ds = KalmanNetDataset(
    split='val',
    seq_len=50,
    stride=25,
    io_checkpoint_path=io_ckpt_str
)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train Sequences: {len(train_ds)} ({len(train_loader)} batches)')
print(f'Val Sequences  : {len(val_ds)} ({len(val_loader)} batches)')

In [ ]:
model = KalmanNetNN(
    state_dim=4,
    meas_dim=2,
    hidden_dim=64,
    num_layers=2,
    dropout=0.1
).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'KalmanNet Initialized. Trainable Parameters: {param_count:,}')

dt = 0.1
F_mat = torch.tensor([
    [1.0, 0.0, dt,  0.0],
    [0.0, 1.0, 0.0, dt ],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
], dtype=torch.float32, device=device)

B_mat = torch.tensor([
    [0.5 * dt**2, 0.0],
    [0.0, 0.5 * dt**2],
    [dt, 0.0],
    [0.0, dt]
], dtype=torch.float32, device=device)

H_mat = torch.tensor([
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
], dtype=torch.float32, device=device)

In [ ]:
def rollout_kalmannet(model, a_nav, z_meas, init_state, F_m, B_m, H_m):
    """
    Differentiable rollout of KalmanNet over a sequence batch.
    """
    B, L, _ = a_nav.shape
    x_curr = init_state
    z_prev = z_meas[:, 0, :]
    h_prev = None

    post_states = []
    gains = []

    for t in range(1, L):
        a_t = a_nav[:, t, :]
        z_t = z_meas[:, t, :]

        # Physics Prior Prediction
        x_prior = torch.matmul(x_curr, F_m.T) + torch.matmul(a_t, B_m.T)

        # KalmanNet Adaptive Filter Step
        x_post, K_gain, h_prev = model.step(
            x_prior=x_prior,
            z_meas=z_t,
            H_matrix=H_m,
            x_prev=x_curr,
            z_prev=z_prev,
            h_prev=h_prev
        )

        post_states.append(x_post)
        gains.append(K_gain)
        x_curr = x_post
        z_prev = z_t

    pred_states = torch.stack(post_states, dim=1)  # (B, L-1, 4)
    stack_gains = torch.stack(gains, dim=1)        # (B, L-1, 4, 2)
    return pred_states, stack_gains

print('Rollout function compiled successfully.')

In [ ]:
EPOCHS = 50
LEARNING_RATE = 1e-3
EARLY_STOP_PATIENCE = 10

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_val_loss    = float('inf')
best_epoch       = 0
patience_counter = 0
best_ckpt_path   = ckpt_dir / 'kalmannet_best.pt'

history = {
    'train_loss': [], 'val_loss': [],
    'train_pos_rmse': [], 'val_pos_rmse': [],
    'train_vel_rmse': [], 'val_vel_rmse': []
}

print(f'Starting KalmanNet Training for up to {EPOCHS} epochs on {device}...')
print(f'NIO source: {nio_source}')
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_losses, t_pos_errs, t_vel_errs = [], [], []

    for batch in train_loader:
        a_nav      = batch['a_nav'].to(device)
        z_meas     = batch['z_meas'].to(device)
        gt_state   = batch['gt_state'].to(device)
        init_state = batch['init_state'].to(device)

        optimizer.zero_grad()
        pred_states, _ = rollout_kalmannet(model, a_nav, z_meas, init_state, F_mat, B_mat, H_mat)
        gt_target = gt_state[:, 1:, :]

        pos_diff = pred_states[:, :, 0:2] - gt_target[:, :, 0:2]
        vel_diff = pred_states[:, :, 2:4] - gt_target[:, :, 2:4]
        loss_pos = torch.mean(pos_diff ** 2)
        loss_vel = torch.mean(vel_diff ** 2)
        loss = loss_pos + 0.5 * loss_vel

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        t_losses.append(loss.item())
        t_pos_errs.append(torch.sqrt(loss_pos).item())
        t_vel_errs.append(torch.sqrt(loss_vel).item())

    scheduler.step()

    model.eval()
    v_losses, v_pos_errs, v_vel_errs = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            a_nav      = batch['a_nav'].to(device)
            z_meas     = batch['z_meas'].to(device)
            gt_state   = batch['gt_state'].to(device)
            init_state = batch['init_state'].to(device)

            pred_states, _ = rollout_kalmannet(model, a_nav, z_meas, init_state, F_mat, B_mat, H_mat)
            gt_target = gt_state[:, 1:, :]

            pos_diff = pred_states[:, :, 0:2] - gt_target[:, :, 0:2]
            vel_diff = pred_states[:, :, 2:4] - gt_target[:, :, 2:4]
            loss_pos = torch.mean(pos_diff ** 2)
            loss_vel = torch.mean(vel_diff ** 2)
            loss = loss_pos + 0.5 * loss_vel

            v_losses.append(loss.item())
            v_pos_errs.append(torch.sqrt(loss_pos).item())
            v_vel_errs.append(torch.sqrt(loss_vel).item())

    mean_t_loss = float(np.mean(t_losses))
    mean_v_loss = float(np.mean(v_losses))
    mean_t_pos  = float(np.mean(t_pos_errs))
    mean_v_pos  = float(np.mean(v_pos_errs))
    mean_t_vel  = float(np.mean(t_vel_errs))
    mean_v_vel  = float(np.mean(v_vel_errs))

    history['train_loss'].append(mean_t_loss)
    history['val_loss'].append(mean_v_loss)
    history['train_pos_rmse'].append(mean_t_pos)
    history['val_pos_rmse'].append(mean_v_pos)
    history['train_vel_rmse'].append(mean_t_vel)
    history['val_vel_rmse'].append(mean_v_vel)

    if mean_v_loss < best_val_loss:
        best_val_loss = mean_v_loss
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'val_pos_rmse': mean_v_pos,
            'val_vel_rmse': mean_v_vel,
            'nio_source': nio_source,
            'nio_checkpoint': io_ckpt_str,
            'results_tag': results_tag,
            'config': {'state_dim': 4, 'meas_dim': 2, 'hidden_dim': 64, 'num_layers': 2}
        }, best_ckpt_path)
        star = ' *** (Best)'
    else:
        patience_counter += 1
        star = ''

    if epoch % 5 == 0 or epoch == 1 or star:
        elapsed = (time.time() - start_time) / 60
        print(
            f'Ep {epoch:02d}/{EPOCHS} | '
            f'Train: {mean_t_loss:.4f} (Pos={mean_t_pos:.2f}m Vel={mean_t_vel:.2f}m/s) | '
            f'Val: {mean_v_loss:.4f} (Pos={mean_v_pos:.2f}m Vel={mean_v_vel:.2f}m/s) | '
            f'{elapsed:.1f}min{star}'
        )

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'[EARLY STOP] No improvement for {EARLY_STOP_PATIENCE} epochs. Best: epoch {best_epoch}')
        break

elapsed = time.time() - start_time
print(f'\nTraining Completed in {elapsed:.1f}s. Best Val Loss: {best_val_loss:.4f} (epoch {best_epoch})')
print(f'Checkpoint: {best_ckpt_path}')

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', color='#1f77b4', lw=2)
plt.plot(history['val_loss'],   label='Val Loss',   color='#ff7f0e', lw=2)
plt.xlabel('Epoch'); plt.ylabel('Trajectory Loss')
plt.title(f'KalmanNet Loss ({results_tag})')
plt.grid(True, alpha=0.3); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_pos_rmse'], label='Train Pos RMSE (m)',   color='#2ca02c', lw=1.8)
plt.plot(history['val_pos_rmse'],   label='Val Pos RMSE (m)',     color='#d62728', lw=1.8)
plt.plot(history['train_vel_rmse'], label='Train Vel RMSE (m/s)', color='#9467bd', lw=1.5, ls='--')
plt.plot(history['val_vel_rmse'],   label='Val Vel RMSE (m/s)',   color='#8c564b', lw=1.5, ls='--')
plt.xlabel('Epoch'); plt.ylabel('RMSE')
plt.title('KalmanNet State Estimation Errors')
plt.grid(True, alpha=0.3); plt.legend()

plt.tight_layout()
curve_path = plots_dir / f'kalmannet_training_curve_{results_tag}.png'
plt.savefig(curve_path, dpi=150); plt.close()
print(f'Training curve saved: {curve_path}')

# Results (to tag-specific dir for clean comparison)
results_dir = PROJECT_ROOT / 'results' / results_tag
results_dir.mkdir(parents=True, exist_ok=True)
metrics = {
    'experiment':          results_tag,
    'nio_source':          nio_source,
    'nio_checkpoint':      io_ckpt_str,
    'best_epoch':          best_epoch,
    'best_val_loss':       round(best_val_loss, 6),
    'final_val_pos_rmse':  round(history['val_pos_rmse'][best_epoch-1], 4),
    'final_val_vel_rmse':  round(history['val_vel_rmse'][best_epoch-1], 4),
    'epochs_trained':      len(history['train_loss']),
    'checkpoint':          str(best_ckpt_path.relative_to(PROJECT_ROOT)),
    'device':              str(device),
}
metrics_path = results_dir / 'kalmannet_training_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved: {metrics_path}')
print(f'\nKey results:')
for k, v in metrics.items():
    if k not in ('nio_checkpoint',):
        print(f'  {k}: {v}')